# Clasificación clásica de lugares para estudiar

Entrena un Random Forest supervisado con características HOG/color de las fotos y metadatos. Lee CSV e imágenes directamente del ZIP, no requiere extraerlo a mano y guarda un artefacto joblib reutilizable.

Las filas inciertas no se usan para entrenar ni evaluar; se predicen por separado. El split determinista coincide con el usado por los experimentos TensorC/DQN.

In [ ]:
from __future__ import annotations

import csv
import io
import json
import random
import time
import zipfile
from pathlib import Path, PurePosixPath

import joblib
import numpy as np
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

ZIP_PATH = Path("Datos-Supabase-2026-11-18.zip")
PROJECT_ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "experiments/study_classifier").is_dir()
)
OUTPUT_DIR = PROJECT_ROOT / "experiments/study_classifier/results/python_classic"
SEED = 42
IMAGE_SIZE = 32
HOG_BINS = 9
MODEL_PATH = OUTPUT_DIR / "study_classifier_random_forest.joblib"

if not ZIP_PATH.is_file():
    raise FileNotFoundError(f"No se encontró el ZIP: {ZIP_PATH}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"ZIP: {ZIP_PATH}")
print(f"Salida: {OUTPUT_DIR.resolve()}")

## Carga y partición

La división reproduce el mismo procedimiento y semilla que el clasificador previo: separación por clase 70/15/15 y mezcla determinista de cada partición.

In [ ]:
def split_rows(rows, seed=SEED):
    rng = random.Random(seed)
    by_label = {"apto": [], "no_apto": []}
    for row in rows:
        if row["study_label"] in by_label and row.get("foto_local"):
            by_label[row["study_label"]].append(row)
    for values in by_label.values():
        rng.shuffle(values)

    splits = {"train": [], "val": [], "test": []}
    for values in by_label.values():
        n_train = int(len(values) * 0.70)
        n_val = int(len(values) * 0.15)
        splits["train"] += values[:n_train]
        splits["val"] += values[n_train:n_train + n_val]
        splits["test"] += values[n_train + n_val:]
    for values in splits.values():
        rng.shuffle(values)

    uncertain = [
        row for row in rows
        if row["study_label"] == "incierto" and row.get("foto_local")
    ]
    return splits, uncertain


archive = zipfile.ZipFile(ZIP_PATH)
csv_member = next(
    (name for name in archive.namelist()
     if name.endswith("/datos.csv") or name == "datos.csv"),
    None,
)
if csv_member is None:
    archive.close()
    raise ValueError("El ZIP no contiene datos.csv")

archive_prefix = csv_member.rsplit("/", 1)[0] if "/" in csv_member else ""
with archive.open(csv_member) as raw_csv:
    text_csv = io.TextIOWrapper(raw_csv, encoding="utf-8-sig", newline="")
    rows = list(csv.DictReader(text_csv))

required_columns = {
    "id", "study_label", "people_count", "has_table", "visual_occupancy",
    "space_type", "temperature_c", "humidity_pct", "sound_dbfs", "foto_local",
}
missing_columns = required_columns.difference(rows[0] if rows else {})
if missing_columns:
    archive.close()
    raise ValueError(f"Faltan columnas en datos.csv: {sorted(missing_columns)}")

splits, uncertain_rows = split_rows(rows)
archive_names = set(archive.namelist())

def zip_photo_member(row):
    relative = PurePosixPath(row["foto_local"])
    if relative.is_absolute() or ".." in relative.parts:
        raise ValueError(f"Ruta de foto inválida: {row['foto_local']}")
    return str(PurePosixPath(archive_prefix) / relative)

for split_name in ("train", "val", "test"):
    missing = [row["id"] for row in splits[split_name]
               if zip_photo_member(row) not in archive_names]
    if missing:
        archive.close()
        raise FileNotFoundError(
            f"{len(missing)} fotos faltan en el split {split_name}: {missing[:3]}"
        )

uncertain_rows = [
    row for row in uncertain_rows
    if zip_photo_member(row) in archive_names
]

split_counts = {name: len(values) for name, values in splits.items()}
print("Filas del ZIP:", len(rows))
print("Split:", split_counts)
print("Inciertas disponibles para predicción:", len(uncertain_rows))
assert split_counts == {"train": 211, "val": 45, "test": 47}

## Características

Cada imagen se reduce a 32×32 RGB. HOG resume bordes y formas; histogramas RGB conservan color. Se concatenan 11 metadatos normalizados con los mismos valores de respaldo del experimento TensorC.

In [ ]:
def metadata_features(row):
    occupancy = {
        "despejado": [1, 0, 0],
        "medio": [0, 1, 0],
        "lleno": [0, 0, 1],
    }.get(row["visual_occupancy"], [0, 0, 0])
    space = {
        "exterior": [1, 0, 0],
        "interior": [0, 1, 0],
        "mixto": [0, 0, 1],
    }.get(row["space_type"], [0, 0, 0])

    people = min(float(row["people_count"] or 0), 30.0) / 30.0
    table = float(row["has_table"] == "True")
    temperature = (float(row["temperature_c"] or 30) - 20.0) / 20.0
    humidity = float(row["humidity_pct"] or 0) / 100.0
    sound = (float(row["sound_dbfs"] or -40) + 40.0) / 40.0
    values = [people, table, *occupancy, *space, temperature, humidity, sound]
    result = np.asarray(values, dtype=np.float32)
    if result.shape != (11,) or not np.isfinite(result).all():
        raise ValueError(f"Metadatos inválidos para id={row['id']}")
    return result


def hog_features(gray):
    gx = np.zeros_like(gray)
    gy = np.zeros_like(gray)
    gx[:, 1:-1] = (gray[:, 2:] - gray[:, :-2]) * 0.5
    gx[:, 0] = gray[:, 1] - gray[:, 0]
    gx[:, -1] = gray[:, -1] - gray[:, -2]
    gy[1:-1, :] = (gray[2:, :] - gray[:-2, :]) * 0.5
    gy[0, :] = gray[1, :] - gray[0, :]
    gy[-1, :] = gray[-1, :] - gray[-2, :]

    magnitude = np.hypot(gx, gy)
    angle = np.rad2deg(np.arctan2(gy, gx)) % 180.0
    bin_index = np.minimum((angle * HOG_BINS / 180.0).astype(np.int32),
                           HOG_BINS - 1)
    cells = np.zeros((4, 4, HOG_BINS), dtype=np.float32)
    for cell_y in range(4):
        for cell_x in range(4):
            y0, x0 = cell_y * 8, cell_x * 8
            bins = bin_index[y0:y0 + 8, x0:x0 + 8].ravel()
            weights = magnitude[y0:y0 + 8, x0:x0 + 8].ravel()
            cells[cell_y, cell_x] = np.bincount(
                bins, weights=weights, minlength=HOG_BINS
            )
    blocks = []
    for block_y in range(3):
        for block_x in range(3):
            block = cells[block_y:block_y + 2, block_x:block_x + 2].ravel()
            block = block / np.sqrt(np.dot(block, block) + 1e-6)
            block = np.minimum(block, 0.2)
            block = block / np.sqrt(np.dot(block, block) + 1e-6)
            blocks.extend(block)
    return np.asarray(blocks, dtype=np.float32)


def image_features(row):
    member = zip_photo_member(row)
    with archive.open(member) as image_file:
        image = Image.open(image_file).convert("RGB")
        image = image.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)
        rgb = np.asarray(image, dtype=np.uint8)

    gray = np.asarray(image.convert("L"), dtype=np.float32) / 255.0
    hog = hog_features(gray)
    color_histograms = np.concatenate([
        np.histogram(rgb[:, :, channel], bins=8, range=(0, 256))[0]
        for channel in range(3)
    ]).astype(np.float32)
    color_histograms /= float(IMAGE_SIZE * IMAGE_SIZE)
    result = np.concatenate([hog, color_histograms, metadata_features(row)])
    if result.shape != (359,) or not np.isfinite(result).all():
        raise ValueError(f"Vector de características inválido para id={row['id']}")
    return result


feature_started = time.perf_counter()
features_by_id = {}
for split_name in ("train", "val", "test"):
    for row in splits[split_name]:
        features_by_id[row["id"]] = image_features(row)
for row in uncertain_rows:
    features_by_id[row["id"]] = image_features(row)
feature_seconds = time.perf_counter() - feature_started
archive.close()

def make_xy(rows_to_use):
    x = np.stack([features_by_id[row["id"]] for row in rows_to_use])
    y = np.asarray([int(row["study_label"] == "apto")
                    for row in rows_to_use], dtype=np.int32)
    return x, y

x_train, y_train = make_xy(splits["train"])
x_val, y_val = make_xy(splits["val"])
x_test, y_test = make_xy(splits["test"])
print(f"Características: {x_train.shape[1]} | "
      f"extracción ZIP: {feature_seconds:.2f} s")

In [ ]:
model = RandomForestClassifier(
    n_estimators=500,
    max_features="sqrt",
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
)

fit_started = time.perf_counter()
model.fit(x_train, y_train)
fit_seconds = time.perf_counter() - fit_started

prob_apto = model.predict_proba(x_test)[:, list(model.classes_).index(1)]
pred_test = (prob_apto >= 0.5).astype(np.int32)
metrics = {
    "accuracy": float(accuracy_score(y_test, pred_test)),
    "precision_apto": float(precision_score(y_test, pred_test, zero_division=0)),
    "recall_apto": float(recall_score(y_test, pred_test, zero_division=0)),
    "f1_apto": float(f1_score(y_test, pred_test, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, prob_apto)),
    "confusion_matrix": confusion_matrix(y_test, pred_test, labels=[0, 1]).tolist(),
    "test_rows": int(len(y_test)),
    "train_rows": int(len(y_train)),
    "feature_count": int(x_train.shape[1]),
    "feature_extraction_seconds": float(feature_seconds),
    "training_seconds": float(fit_seconds),
}
print(json.dumps(metrics, indent=2))

In [ ]:
artifact = {
    "model": model,
    "feature_config": {
        "image_size": IMAGE_SIZE,
        "hog_bins": HOG_BINS,
        "hog_cells": [4, 4],
        "rgb_histogram_bins": 8,
        "metadata_features": [
            "people_count", "has_table", "visual_occupancy",
            "space_type", "temperature_c", "humidity_pct", "sound_dbfs",
        ],
        "class_names": {0: "no_apto", 1: "apto"},
        "split_seed": SEED,
    },
}
joblib.dump(artifact, MODEL_PATH)
(OUTPUT_DIR / "metrics.json").write_text(
    json.dumps(metrics, indent=2), encoding="utf-8"
)

def save_predictions(path, rows_to_use, probabilities):
    predicted = (probabilities >= 0.5).astype(np.int32)
    with path.open("w", newline="", encoding="utf-8") as output_file:
        writer = csv.writer(output_file)
        writer.writerow(["id", "real", "prediccion", "prob_apto"])
        for row, prediction, probability in zip(
            rows_to_use, predicted, probabilities
        ):
            writer.writerow([
                row["id"], row["study_label"],
                "apto" if prediction else "no_apto", f"{probability:.7f}",
            ])

save_predictions(OUTPUT_DIR / "test_predictions.csv", splits["test"], prob_apto)
uncertain_probabilities = (
    model.predict_proba(np.stack([
        features_by_id[row["id"]] for row in uncertain_rows
    ]))[:, list(model.classes_).index(1)]
    if uncertain_rows else np.asarray([], dtype=np.float32)
)
save_predictions(
    OUTPUT_DIR / "uncertain_predictions.csv",
    uncertain_rows,
    uncertain_probabilities,
)
print(f"Modelo guardado: {MODEL_PATH.resolve()}")
print(f"Métricas: {(OUTPUT_DIR / 'metrics.json').resolve()}")
print(f"Predicciones inciertas: {len(uncertain_rows)}")

In [ ]:
saved_artifact = joblib.load(MODEL_PATH)
reloaded = saved_artifact["model"]
reloaded_predictions = reloaded.predict(x_test)
assert np.array_equal(reloaded_predictions, pred_test)
print("Recarga verificada: las predicciones coinciden.")